In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

if os.environ['OPENAI_API_KEY']:
    print("OPENAI_API_KEY is set")

OPENAI_API_KEY is set


In [2]:
from langchain_openai import ChatOpenAI

In [3]:
llm = ChatOpenAI(model= 'gpt-5-nano', temperature=0)


In [4]:
response = llm.invoke('Hello, how are you?')
response.content

'Hi there! I’m here and ready to help. I don’t have feelings, but I’m functioning well. How can I assist you today? If you’d like, tell me what you’re curious about or what you need help with.'

## Adding more data using pdf

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./rdr details.pdf")

docs = loader.load()

docs

[Document(metadata={'producer': 'calibre (4.6.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.6.0) [https://calibre-ebook.com]', 'creationdate': '2019-12-15T02:04:58+00:00', 'source': './rdr details.pdf', 'file_path': './rdr details.pdf', 'total_pages': 319, 'format': 'PDF 1.4', 'title': 'Red Dead Redemption', 'author': 'Rockstar Games', 'subject': '', 'keywords': '', 'moddate': '2019-12-14T21:04:58-05:00', 'trapped': '', 'modDate': "D:20191214210458-05'00'", 'creationDate': "D:20191215020458+00'00'", 'page': 0}, page_content=''),
 Document(metadata={'producer': 'calibre (4.6.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.6.0) [https://calibre-ebook.com]', 'creationdate': '2019-12-15T02:04:58+00:00', 'source': './rdr details.pdf', 'file_path': './rdr details.pdf', 'total_pages': 319, 'format': 'PDF 1.4', 'title': 'Red Dead Redemption', 'author': 'Rockstar Games', 'subject': '', 'keywords': '', 'moddate': '2019-12-14T21:04:58-05:00', 'trapped': '', 'modDate': "D:2019121

In [8]:
for i in docs:
    i.metadata = {"source": 'rdr details',
                  'developer':  'John Doe',
    }
    

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(docs)

len(chunks)

318

In [12]:
chunks[0].metadata

{'source': 'rdr details', 'developer': 'John Doe'}

### Store embeddings in the existing local vector

In [22]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

In [23]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
        persist_directory = "./VectorDB",
        embedding_function = embedding
)

In [24]:
vectorstore.add_documents(chunks)

['03df7a60-db78-49dd-ba5a-b69a25c78626',
 'd45c2a27-e602-42e1-93c2-2b3244ecb6a3',
 'e0a96a40-c432-44dd-9755-70bd80d3b124',
 '5d5b2e55-7fee-43ca-b068-958562647127',
 '3bd28377-56e2-4c98-896c-58ebe03cbd94',
 '1fef6a49-91ec-4c39-9483-9f844c82148a',
 '2508345f-d6c9-4121-8a96-baf700eff2e2',
 'a9ce3685-b04f-4242-9999-01fc14cd0a06',
 '15fc8800-a764-4555-8409-241c76271a10',
 '85238e30-e978-4d08-a9b4-96874fd32b48',
 '59e840e7-2292-4b3c-afc5-224e48d9e21a',
 '8f9c4c3c-dade-4173-9424-3586ea289651',
 'da052ac6-d8c1-4a80-a100-beaf59d973e4',
 '922e033a-f399-4a80-9b8d-9b47bdae3beb',
 '19f00841-53b7-48be-984b-13cae478318e',
 'f12c365a-8b56-4b10-b8d9-419c30481f67',
 '821ecc06-91d0-433f-9b9e-7a52d6b3b3db',
 '3c2d43c3-ae4b-41bd-a466-fa45dd50a1cd',
 '06513330-9df0-4031-89c9-aaff7c2729ad',
 '9f37915f-7e3b-439a-afff-b8dba9852ece',
 'b57c60ba-48b2-40f1-b0c8-d2f104d4b8b8',
 '2627305e-69f5-4fd0-9117-d5cabe664a06',
 '108201c8-e0d2-45bd-85c4-bd3b758ef808',
 '293ef569-1454-4155-8802-7da612f8ede0',
 '0238c3cb-2263-

In [27]:
context = vectorstore.similarity_search("tell me about spider orchird", k=3)

context

[Document(metadata={'source': 'rdr details', 'developer': 'John Doe'}, page_content='Spider Orchid\nBy\nArthur Morgan\n \nSpider Orchids can be identified by the long sepals that resemble the\nlegs of a spider. The flowers blossom in shades of cream, green,\nbrownish red, and spotted yellow. These flowers can often be found\nin swamp lands or damp forest grounds in Bayou Nwa, Lemoyne.\nThe orchidelirium trend has meant that this rare orchid is extremely\nvaluable to collectors and botanists.'),
 Document(metadata={'source': 'rdr details', 'developer': 'John Doe'}, page_content='Ghost Orchid\nBy\nArthur Morgan\n \nThe Ghost Orchid is named after the white to pale pink flowers that\ncan be seen in shaded habitats. This small orchid grows in shaded\nareas, such as woodlands, and under leaves and soil, with only the\nflower appearing above ground. The Ghost Orchid is native to Bayou\nNwa, Lemoyne. The orchidelirium trend has meant that this rare\norchid is extremely valuable to collectors 

In [29]:
response = llm.invoke('tell me about spider orchird?')
response.content

'“Spider orchid” is a common name used for several orchids that have long, threadlike sepals and petals arranged in a way that makes the flower look spider‑like. The group most people mean are the Australian spider orchids, which belong mainly to the genus Caladenia. There are many species with different colors and sizes, but they share the characteristic spidery, delicate appearance.\n\nKey points:\n- Appearance: flowers with slender, hanging or spreading sepals and petals; a prominent labellum (the lip) that’s often brightly colored or patterned to attract pollinators.\n- Where they’re from: primarily native to Australia (with some related species in nearby regions). They grow in a variety of habitats, from woodland and open forest to heathland.\n- Bloom time: usually spring to early summer (in the Southern Hemisphere); exact timing depends on the species and local climate.\n- Ecology: terrestrial orchids that grow from underground tubers. They rely on specific mycorrhizal fungi to g

In [30]:
response = llm.invoke(f'tell me about spider orchird? Please use this context:{context}')
response.content

'Spider Orchid (spelled correctly)\n\n- Identification: Long sepals that resemble spider legs.\n- Flowers: Shades of cream, green, brownish red, and spotted yellow.\n- Habitat: Often found in swamp lands or damp forest grounds in Bayou Nwa, Lemoyne.\n- Rarity and value: Considered rare; the orchidelirium trend has made it extremely valuable to collectors and botanists.\n\nSource of this information: Arthur Morgan, Spider Orchid, from the rdr details documents.'